# AWS Bedrock AgentCore — LangGraph Agent

This notebook deploys a LangGraph ReAct agent to Bedrock AgentCore Runtime using the **HTTP protocol**. The agent has **no local tools** — it connects to the AgentCore-hosted MCP math server and uses its async tools via `langchain-mcp-adapters`.

### Architecture
```
Client → AgentCore (HTTP) → LangGraph Agent → AgentCore (MCP) → MCP Math Server
```

### Tests
- Invoke the agent in parallel with unique sessions
- Invoke the agent in parallel with a shared session
- Compare latency and server instance distribution

## Install Dependencies

In [1]:
#!pip install --force-reinstall -U -r requirements.txt --quiet

## Setup

In [7]:
from bedrock_agentcore_starter_toolkit import Runtime
from bedrock_agentcore_starter_toolkit.operations.runtime import destroy_bedrock_agentcore
from boto3.session import Session
from pathlib import Path
import os

boto_session = Session()
region = boto_session.region_name

agentcore_control_client = boto_session.client("bedrock-agentcore-control", region_name=region)
ssm_client = boto_session.client("ssm", region_name=region)

agent_name = "blog_langgraph_agent"

print(f"Using AWS region: {region}")

Using AWS region: us-west-2


## Deploy AgentCore LangGraph Agent

Configure and launch the LangGraph agent using the **HTTP** protocol. The agent exposes a `POST /invocations` endpoint that accepts `{"prompt": "..."}` payloads. It connects to the MCP math server to use its async tools.

In [8]:
# Configure and launch the LangGraph agent
agentcore_runtime = Runtime()
response = agentcore_runtime.configure(
    entrypoint="agent.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    region=region,
    protocol="HTTP",
    agent_name=agent_name,
)

launch_result = agentcore_runtime.launch()
print(f"Launch completed. Agent ARN: {launch_result.agent_arn}")

# Store the agent ARN in Parameter Store for the test script
ssm_client.put_parameter(
    Name="/blog_langgraph_agent/runtime_iam/agent_arn",
    Value=launch_result.agent_arn,
    Type="String",
    Description="Agent ARN for blog_langgraph_agent",
    Overwrite=True,
)

Entrypoint parsed: file=/Users/mingyyzz/dev-local/aws.blog1/src/langgraph_agent/agent.py, bedrock_agentcore_name=agent
Memory disabled - agent will be stateless
Configuring BedrockAgentCore agent: blog_langgraph_agent
Memory disabled
Network mode: PUBLIC


📄 Using existing Dockerfile: /Users/mingyyzz/dev-local/aws.blog1/src/langgraph_agent/Dockerfile

Generated .dockerignore: /Users/mingyyzz/dev-local/aws.blog1/src/langgraph_agent/.dockerignore
Keeping 'blog_langgraph_agent' as default agent
Bedrock AgentCore configured: /Users/mingyyzz/dev-local/aws.blog1/src/langgraph_agent/.bedrock_agentcore.yaml
🚀 Launching Bedrock AgentCore (cloud mode - RECOMMENDED)...
   • Deploy Python code directly to runtime
   • No Docker required (DEFAULT behavior)
   • Production-ready deployment

💡 Deployment options:
   • runtime.launch()                → Cloud (current)
   • runtime.launch(local=True)      → Local development
Memory disabled - skipping memory creation
Starting CodeBuild ARM64 deployment for agent 'blog_langgraph_agent' to account 980413094772 (us-west-2)
Generated image tag: 20260331-232941-642
Setting up AWS resources (ECR repository, execution roles)...
Getting or creating ECR repository for agent: blog_langgraph_agent
ECR repository available: 980413094772.dkr.ecr.us-west-2.amazonaws.com/bedrock-agentcore-blog_langgraph_agent
Gett

✅ Reusing existing ECR repository: 980413094772.dkr.ecr.us-west-2.amazonaws.com/bedrock-agentcore-blog_langgraph_agent


✅ Reusing existing execution role: arn:aws:iam::980413094772:role/AmazonBedrockAgentCoreSDKRuntime-us-west-2-13dd8d44ac
Execution role available: arn:aws:iam::980413094772:role/AmazonBedrockAgentCoreSDKRuntime-us-west-2-13dd8d44ac
Preparing CodeBuild project and uploading source...
Getting or creating CodeBuild execution role for agent: blog_langgraph_agent
Role name: AmazonBedrockAgentCoreSDKCodeBuild-us-west-2-13dd8d44ac
Reusing existing CodeBuild execution role: arn:aws:iam::980413094772:role/AmazonBedrockAgentCoreSDKCodeBuild-us-west-2-13dd8d44ac
Using dockerignore.template with 47 patterns for zip filtering
Uploaded source to S3: blog_langgraph_agent/source.zip
Updated CodeBuild project: bedrock-agentcore-blog_langgraph_agent-builder
Starting CodeBuild build (this may take several minutes)...
Starting CodeBuild monitoring...
🔄 QUEUED started (total: 0s)
✅ QUEUED completed in 1.0s
🔄 PROVISIONING started (total: 1s)
✅ PROVISIONING completed in 7.3s
🔄 DOWNLOAD_SOURCE started (total: 

Launch completed. Agent ARN: arn:aws:bedrock-agentcore:us-west-2:980413094772:runtime/blog_langgraph_agent-79UgbT8kD3


{'Version': 3,
 'Tier': 'Standard',
 'ResponseMetadata': {'RequestId': '72fe9a96-106c-45c4-911b-e3366a42db91',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'server': 'Server',
   'date': 'Tue, 31 Mar 2026 23:30:23 GMT',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '31',
   'connection': 'keep-alive',
   'x-amzn-requestid': '72fe9a96-106c-45c4-911b-e3366a42db91',
   'cache-control': 'no-store'},
  'RetryAttempts': 0}}

## Test: 5 Calls, Unique Sessions

Each call gets its own session on the LangGraph agent, so each is routed to a separate warm pool instance. The agent internally calls the MCP server's async tools.

In [10]:
!python test_agent.py --calls 5 --session unique


langgraph agent — 5 concurrent calls, unique sessions
  agent #0   start=18:03:07, end=18:03:12, duration=  5.78s, server=e0221798-df2a-44aa-9b87-3e14b47d7530  result=result=0 server=8e6c0753-c447-4c5e-9d42-6273078664b1
  agent #4   start=18:03:07, end=18:03:13, duration=  6.40s, server=e3f19673-3401-43df-95be-195d5d6db6a6  result=result=8 server=2df745e3-7b5a-44fe-b53b-3d7a7cf62401
  agent #2   start=18:03:07, end=18:03:13, duration=  6.52s, server=ab44a517-c4a8-4191-b19f-5290c1089d21  result=result=4 server=87c9d33a-b70a-45e3-beba-c9c770331366
  agent #3   start=18:03:07, end=18:03:15, duration=  7.93s, server=fd3eb0e9-96f0-4d93-a677-a7cc18cc67aa  result=result=6 server=444cac76-0a27-4c83-a7b2-9f28f73a002d
  agent #1   start=18:03:07, end=18:03:15, duration=  8.21s, server=9781c1f3-db19-4406-b907-784d553d3acf  result=result=2 server=c2b6fe85-09a7-4fee-a4fc-1c35a40decb4
  total wall time: 8.29s  avg per call: 6.97s
  unique server instances: 5


## Cleanup (Optional)

In [ ]:
# Uncomment to clean up resources
# try:
#     ssm_client.delete_parameter(Name="/blog_langgraph_agent/runtime_iam/agent_arn")
#     print("Parameter Store parameter deleted")
# except ssm_client.exceptions.ParameterNotFound:
#     print("Parameter Store parameter not found")

# destroy_bedrock_agentcore(
#     config_path=Path(".bedrock_agentcore.yaml"),
#     agent_name=agent_name,
#     delete_ecr_repo=True,
# )